In [1]:
# ============================================================
# LORD OF THE RINGS - NLP + K-MEANS CLUSTERING
# Dataset: LOTR 2.csv
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import warnings

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


# ============================================================
# 2. LOAD DATASET
# ============================================================

file_name = "LOTR 2.csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("\nDataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 3. BASIC DATASET INFORMATION
# ============================================================

print("\n" + "="*60)
print("DATASET INFORMATION")
print("="*60)

print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# ============================================================
# 4. REMOVE DUPLICATE ROWS
# ============================================================

df = df.drop_duplicates().reset_index(drop=True)

print("\nDataset shape after removing duplicates:", df.shape)


# ============================================================
# 5. IDENTIFY TEXT COLUMN
# ============================================================

# We try to automatically identify the column containing dialogue/text.

possible_text_columns = [
    "dialogue",
    "Dialogue",
    "text",
    "Text",
    "quote",
    "Quote",
    "line",
    "Line",
    "speech",
    "Speech",
    "dialog",
    "Dialog"
]

text_column = None

for column in possible_text_columns:
    if column in df.columns:
        text_column = column
        break


# If no common text column is found,
# automatically select the object/string column with the
# greatest average text length.

if text_column is None:

    object_columns = df.select_dtypes(
        include=["object"]
    ).columns.tolist()

    if len(object_columns) == 0:
        raise ValueError(
            "No text column was found in the dataset."
        )

    average_lengths = {}

    for column in object_columns:
        average_lengths[column] = (
            df[column]
            .fillna("")
            .astype(str)
            .str.len()
            .mean()
        )

    text_column = max(
        average_lengths,
        key=average_lengths.get
    )


print("\nSelected text column:", text_column)


# ============================================================
# 6. REMOVE MISSING TEXT
# ============================================================

df = df.dropna(
    subset=[text_column]
).copy()

df[text_column] = df[text_column].astype(str)

# Remove empty text rows
df = df[
    df[text_column].str.strip() != ""
].reset_index(drop=True)

print("\nRows after removing empty text:", len(df))


# ============================================================
# 7. NLP TEXT CLEANING
# ============================================================

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove punctuation and numbers
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Strip leading/trailing spaces
    text = text.strip()

    return text


df["clean_text"] = df[text_column].apply(
    clean_text
)


# ============================================================
# 8. DISPLAY ORIGINAL VS CLEANED TEXT
# ============================================================

print("\nOriginal and cleaned text:")

display(
    df[
        [text_column, "clean_text"]
    ].head(10)
)


# ============================================================
# 9. REMOVE VERY SHORT TEXT
# ============================================================

# Remove rows where the cleaned text is extremely short.

df = df[
    df["clean_text"].str.split().str.len() >= 2
].reset_index(drop=True)

print(
    "\nRows after removing very short text:",
    len(df)
)


# ============================================================
# 10. TF-IDF VECTORIZATION
# ============================================================

print("\n" + "="*60)
print("TF-IDF VECTORIZATION")
print("="*60)

vectorizer = TfidfVectorizer(

    # Remove common English stop words
    stop_words="english",

    # Maximum number of vocabulary terms
    max_features=5000,

    # Ignore words appearing in fewer than 2 documents
    min_df=2,

    # Ignore words appearing in more than 95% of documents
    max_df=0.95,

    # Use single words and two-word combinations
    ngram_range=(1, 2)
)


X = vectorizer.fit_transform(
    df["clean_text"]
)

print(
    "TF-IDF matrix shape:",
    X.shape
)

print(
    "Number of features:",
    len(vectorizer.get_feature_names_out())
)


# ============================================================
# 11. GET TF-IDF FEATURE NAMES
# ============================================================

terms = vectorizer.get_feature_names_out()

print("\nFirst 50 TF-IDF features:")

print(
    terms[:50]
)


# ============================================================
# 12. FIND BEST NUMBER OF K-MEANS CLUSTERS
# ============================================================

print("\n" + "="*60)
print("FINDING BEST NUMBER OF CLUSTERS")
print("="*60)

# Test K values from 2 to 10

k_values = range(2, 11)

inertia_values = []

silhouette_values = []


for k in k_values:

    kmeans_test = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans_test.fit_predict(X)

    # Inertia
    inertia_values.append(
        kmeans_test.inertia_
    )

    # Silhouette score
    score = silhouette_score(
        X,
        labels
    )

    silhouette_values.append(score)

    print(
        f"K = {k:2d} | "
        f"Inertia = {kmeans_test.inertia_:.2f} | "
        f"Silhouette = {score:.4f}"
    )


# ============================================================
# 13. ELBOW METHOD GRAPH
# ============================================================

plt.figure(figsize=(9, 6))

plt.plot(
    k_values,
    inertia_values,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.title(
    "Elbow Method for K-Means"
)

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.show()


# ============================================================
# 14. SILHOUETTE SCORE GRAPH
# ============================================================

plt.figure(figsize=(9, 6))

plt.plot(
    k_values,
    silhouette_values,
    marker="o"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.show()


# ============================================================
# 15. SELECT BEST K
# ============================================================

best_k = k_values[
    np.argmax(silhouette_values)
]

best_score = max(
    silhouette_values
)

print("\nBest number of clusters:", best_k)

print(
    "Best silhouette score:",
    round(best_score, 4)
)


# ============================================================
# 16. TRAIN FINAL K-MEANS MODEL
# ============================================================

print("\n" + "="*60)
print("FINAL K-MEANS MODEL")
print("="*60)

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

print(
    "K-Means clustering completed!"
)


# ============================================================
# 17. CLUSTER COUNTS
# ============================================================

print("\nNumber of records in each cluster:")

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print(cluster_counts)


# ============================================================
# 18. CLUSTER DISTRIBUTION GRAPH
# ============================================================

plt.figure(figsize=(9, 6))

cluster_counts.plot(
    kind="bar"
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Dialogue Records"
)

plt.title(
    "Number of LOTR Dialogue Records per Cluster"
)

plt.xticks(
    rotation=0
)

plt.grid(
    axis="y"
)

plt.show()


# ============================================================
# 19. FIND IMPORTANT WORDS IN EACH CLUSTER
# ============================================================

print("\n" + "="*60)
print("IMPORTANT WORDS IN EACH CLUSTER")
print("="*60)


order_centroids = (
    kmeans.cluster_centers_
    .argsort()[:, ::-1]
)


for cluster_number in range(best_k):

    print(
        f"\nCluster {cluster_number}"
    )

    print(
        "-" * 50
    )

    top_words = [
        terms[index]
        for index in
        order_centroids[
            cluster_number,
            :20
        ]
    ]

    print(
        ", ".join(top_words)
    )


# ============================================================
# 20. CREATE TABLE OF TOP WORDS
# ============================================================

top_words_data = []


for cluster_number in range(best_k):

    top_words = [
        terms[index]
        for index in
        order_centroids[
            cluster_number,
            :20
        ]
    ]

    top_words_data.append({

        "Cluster": cluster_number,

        "Top_Words": ", ".join(
            top_words
        )

    })


top_words_df = pd.DataFrame(
    top_words_data
)

print("\nTop words by cluster:")

display(
    top_words_df
)


# ============================================================
# 21. SHOW EXAMPLE DIALOGUE FROM EACH CLUSTER
# ============================================================

print("\n" + "="*60)
print("EXAMPLE DIALOGUE FROM EACH CLUSTER")
print("="*60)


for cluster_number in range(best_k):

    print(
        f"\n\nCLUSTER {cluster_number}"
    )

    print(
        "=" * 60
    )

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    examples = cluster_data[
        text_column
    ].head(10)

    for i, text in enumerate(
        examples,
        start=1
    ):

        print(
            f"{i}. {text}"
        )


# ============================================================
# 22. PCA FOR VISUALISATION
# ============================================================

print("\n" + "="*60)
print("PCA VISUALISATION")
print("="*60)

# Convert TF-IDF sparse matrix to dense matrix
X_dense = X.toarray()

# Reduce to 2 dimensions
pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X_dense
)

print(
    "Explained variance ratio:",
    pca.explained_variance_ratio_
)

print(
    "Total explained variance:",
    round(
        pca.explained_variance_ratio_.sum(),
        4
    )
)


# ============================================================
# 23. PLOT K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(11, 8))

scatter = plt.scatter(

    X_pca[:, 0],

    X_pca[:, 1],

    c=df["cluster"],

    alpha=0.6,

    s=40
)

plt.xlabel(
    "PCA Component 1"
)

plt.ylabel(
    "PCA Component 2"
)

plt.title(
    "LOTR Dialogue Clusters using NLP + K-Means"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(True)

plt.show()


# ============================================================
# 24. ADD PCA VALUES TO DATASET
# ============================================================

df["PCA_1"] = X_pca[:, 0]

df["PCA_2"] = X_pca[:, 1]


# ============================================================
# 25. DISPLAY FINAL DATASET
# ============================================================

print("\n" + "="*60)
print("FINAL DATASET")
print("="*60)

display(
    df.head(20)
)


# ============================================================
# 26. ANALYSE EACH CLUSTER
# ============================================================

print("\n" + "="*60)
print("CLUSTER SUMMARY")
print("="*60)


for cluster_number in range(best_k):

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    print(
        f"\nCluster {cluster_number}"
    )

    print(
        "Number of records:",
        len(cluster_data)
    )

    print(
        "Percentage:",
        round(
            len(cluster_data)
            / len(df)
            * 100,
            2
        ),
        "%"
    )


# ============================================================
# 27. IF CHARACTER COLUMN EXISTS
# ============================================================

# Automatically look for a character column.

possible_character_columns = [

    "character",
    "Character",
    "char",
    "Char",
    "speaker",
    "Speaker",
    "name",
    "Name"

]

character_column = None

for column in possible_character_columns:

    if column in df.columns:

        character_column = column

        break


if character_column is not None:

    print("\n" + "="*60)

    print(
        "CHARACTER DISTRIBUTION BY CLUSTER"
    )

    print("="*60)

    for cluster_number in range(best_k):

        print(
            f"\nCluster {cluster_number}"
        )

        characters = (

            df[
                df["cluster"]
                == cluster_number
            ][character_column]

            .value_counts()

            .head(10)

        )

        display(
            characters
        )

else:

    print(
        "\nNo character column was automatically detected."
    )


# ============================================================
# 28. SAVE RESULTS
# ============================================================

output_file = (
    "LOTR_2_NLP_KMeans_Results.csv"
)

df.to_csv(
    output_file,
    index=False
)

print(
    "\nResults saved to:",
    output_file
)


# ============================================================
# 29. SAVE TOP WORDS
# ============================================================

top_words_file = (
    "LOTR_2_Cluster_Top_Words.csv"
)

top_words_df.to_csv(
    top_words_file,
    index=False
)

print(
    "Top cluster words saved to:",
    top_words_file
)


# ============================================================
# 30. FINAL RESULTS
# ============================================================

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

print(
    "\nDataset:",
    file_name
)

print(
    "Text column:",
    text_column
)

print(
    "Number of records:",
    len(df)
)

print(
    "Number of TF-IDF features:",
    X.shape[1]
)

print(
    "Best K:",
    best_k
)

print(
    "Silhouette Score:",
    round(best_score, 4)
)

print(
    "\nCluster sizes:"
)

print(
    df["cluster"]
    .value_counts()
    .sort_index()
)

print(
    "\nOutput file:",
    output_file
)

print(
    "\nAnalysis completed successfully!"
)

Libraries imported successfully!
Dataset loaded successfully!

Dataset shape: (5, 3)

Column names:
['FellowshipID', 'FirstName', 'Age']

First 5 rows:


,FellowshipID,FirstName,Age
0,1001,Frodo,50
1,1002,Samwise,39
2,1006,Legolas,2931
3,1007,Elrond,6520
4,1008,Barromir,51



DATASET INFORMATION

Number of rows: 5
Number of columns: 3

Data types:
FellowshipID    int64
FirstName         str
Age             int64
dtype: object

Missing values:
FellowshipID    0
FirstName       0
Age             0
dtype: int64

Duplicate rows: 0

Dataset shape after removing duplicates: (5, 3)

Selected text column: FirstName

Rows after removing empty text: 5

Original and cleaned text:


,FirstName,clean_text
0,Frodo,frodo
1,Samwise,samwise
2,Legolas,legolas
3,Elrond,elrond
4,Barromir,barromir



Rows after removing very short text: 0

TF-IDF VECTORIZATION


ValueError: empty vocabulary; perhaps the documents only contain stop words